# 1. Project Overview

# PrettyPlease — AI Beauty Recommendation System

## Project Overview

PrettyPlease is an intelligent beauty product recommendation system designed
to recommend relevant beauty products to new users based on their stated
preferences.

The system combines content-based recommendation, semantic similarity,
product metadata, and a RAG-based explanation layer.

### Current Approach

Since this is a cold-start recommendation scenario with no historical
user interaction data, the system uses content-based recommendation rather
than collaborative filtering.

### Dataset
Link: https://www.kaggle.com/datasets/susant4learning/nykaacosmeticsproductsreview2021

The system uses a Nykaa cosmetics product dataset containing product
metadata, descriptions, categories, ingredients, ratings, and review counts.

## 2. Dataset Loading

The dataset contains 625 Nykaa products with 18 attributes describing
product information, categorization, pricing, descriptions, ingredients,
and customer ratings.

In [ ]:
import pandas as pd

df = pd.read_csv("data/Nykaa_Product_Review.csv")

print(df.shape)
print(df.columns.tolist())

(625, 18)
['Product Id', 'Product Brand Code', 'Retailer', 'Product Category', 'Product Brand', 'Product Name', 'Product Price', 'Product Url', 'Market', 'Product Description', 'Product Currency', 'Product Image Url', 'Product Tags', 'Product Contents', 'Product Rating', 'Product Reviews Count', 'Expected Category Count', 'Expected Brand Count']


In [7]:
df.columns.tolist()

['Product Id',
 'Product Brand Code',
 'Retailer',
 'Product Category',
 'Product Brand',
 'Product Name',
 'Product Price',
 'Product Url',
 'Market',
 'Product Description',
 'Product Currency',
 'Product Image Url',
 'Product Tags',
 'Product Contents',
 'Product Rating',
 'Product Reviews Count',
 'Expected Category Count',
 'Expected Brand Count']

## 3. Dataset Understanding

Before building the recommendation system, we first inspect the structure
and quality of the dataset to understand which attributes can contribute
to product representation and recommendation.

In [8]:
print(f"Number of products: {df.shape[0]}")
print(f"Number of features: {df.shape[1]}")

Number of products: 625
Number of features: 18


## 4. Data Types

We inspect the data types of each attribute to identify numerical,
categorical, and textual features.

In [9]:
df.dtypes

Product Id                  object
Product Brand Code          object
Retailer                    object
Product Category            object
Product Brand               object
Product Name                object
Product Price               object
Product Url                 object
Market                      object
Product Description         object
Product Currency            object
Product Image Url           object
Product Tags                object
Product Contents            object
Product Rating              object
Product Reviews Count      float64
Expected Category Count     object
Expected Brand Count         int64
dtype: object

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 625 entries, 0 to 624
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Product Id               625 non-null    object 
 1   Product Brand Code       625 non-null    object 
 2   Retailer                 625 non-null    object 
 3   Product Category         544 non-null    object 
 4   Product Brand            625 non-null    object 
 5   Product Name             625 non-null    object 
 6   Product Price            625 non-null    object 
 7   Product Url              610 non-null    object 
 8   Market                   625 non-null    object 
 9   Product Description      625 non-null    object 
 10  Product Currency         625 non-null    object 
 11  Product Image Url        625 non-null    object 
 12  Product Tags             610 non-null    object 
 13  Product Contents         324 non-null    object 
 14  Product Rating           5

In [11]:
# a lot of numeric columns are actually categorical bcz of symbols, commas, or inconsistent formatting, so we will fix them during cleaning.

## 5. Category Analysis

In [12]:
df["Product Category"].value_counts().head(30)

Product Category
Makeup > Lips > Lipstick                              22
Mom & Baby > Maternity Wear > Maternity Bra           19
Makeup > Face > Blush                                 15
nykaa.com                                             15
Makeup > Lips > Liquid Lipstick                       14
Makeup > Nails > Nail Polish                          13
Makeup > Face > Foundation                            12
Skin > Moisturizers > Face Moisturizer & Day Cream    12
Makeup > Eyes > Under Eye Concealer                   10
Personal Care > Bath & Shower > Shampoo               10
Makeup > Face > Highlighters                           9
Skin > Body Care > Lotions & Creams                    9
Skin > Shop By Concern > Skin Dryness                  9
Natural > Types of Skin > Normal Skin                  8
Natural > Types of Skin > Dry Skin                     8
Makeup > Face > Concealer                              8
Natural > Hair > Shampoo & Cleanser                    8
Makeup > Combo

In [13]:
df["Product Category"].nunique()

171

In [14]:
df["Product Category"].str.split(">").str[-1].str.strip().value_counts().head(30)

Product Category
Lipstick                        26
Maternity Bra                   19
Blush                           15
nykaa.com                       15
Nail Polish                     14
Shampoo                         14
Liquid Lipstick                 14
Lotions & Creams                13
Face Moisturizer & Day Cream    12
Foundation                      12
Under Eye Concealer             10
Combos @ Nykaa                   9
Skin Dryness                     9
Concealer                        9
Highlighters                     9
Conditioner                      9
Gifts @ Nykaa                    8
Shampoo & Cleanser               8
Normal Skin                      8
Dry Skin                         8
Facewash                         7
Massage Oils                     7
Skin Brightening                 7
Perfumes (EDP/EDT)               7
Perfumes (EDT & EDP)             7
Himalaya                         7
Moisturizer                      7
Lakme                            7
Eye

In [15]:
# 171 category types for 625 products is a lot.
# Natural > Types of Skin > Dry Skin can give us skin_type = dry
# So rather than treating Product Category as one categorical variable, we're going to parse it into meaningful attributes.

In [16]:
# let's inspect the garbage first, nykaa.com is appearing 15 times as a category.
df[df["Product Category"] == "nykaa.com"][[
    "Product Name",
    "Product Category",
    "Product Description"
]]

,Product Name,Product Category,Product Description
36,Seki Edge,nykaa.com,IN
71,Corioliss,nykaa.com,IN
99,Carbamide Forte,nykaa.com,IN
121,Kiko Milano,nykaa.com,IN
145,Organice,nykaa.com,IN
161,Organic India,nykaa.com,IN
179,Wet n Wild,nykaa.com,IN
187,Esbeda,nykaa.com,IN
312,Colorbar,nykaa.com,IN
339,Inveda,nykaa.com,IN


In [17]:
# least common categories, which are also likely garbage.
df["Product Category"].value_counts().tail(30)

Product Category
Hair > Hair Styling > Hair Color                         1
Nykaa Luxe > Makeup > Face > Bronzer                     1
Skin > Kits & Combos > Facial Kits                       1
Mom & Baby > Kids Care > Kids Makeup                     1
Nykaa Luxe > Fragrance > Gifts                           1
Mom & Baby > Baby Care > Wipes & Buds                    1
Natural > Combos @ Nykaa                                 1
Natural > Skin > Face Wash                               1
Health & Wellness > Sexual Wellness > Condoms            1
Brand > VLCC                                             1
Natural > Skin > Moisturizer                             1
Nykaa Luxe > Makeup > Face > Primer                      1
Nykaa Luxe > Makeup > Eyes > Mascara                     1
Mom & Baby > Health & Safety > Detergents & Cleansers    1
Natural > Makeup > Foundation & Concealer                1
Makeup > Eyes > Eye Makeup Remover                       1
Health & Wellness > Shop By Concern > I

## 6. Data Cleaning

The original dataset contains products across several Nykaa categories,
including beauty, personal care, wellness, and unrelated products.
Since PrettyPlease focuses on beauty recommendations, we filter the
dataset to retain relevant beauty and personal-care products.

In [18]:
df["Product Category"].dropna().str.split(">").str[0].str.strip().value_counts()

Product Category
Makeup                            165
Natural                            98
Skin                               73
Personal Care                      35
Brand                              30
Nykaa Luxe                         30
Mom & Baby                         25
Hair                               21
Health & Wellness                  19
Men's Store                        18
nykaa.com                          15
Fragrance                          11
NFBA 2020 Nominees Online Sale      3
Appliances                          1
Name: count, dtype: int64

### 6.1 Filtering for Beauty Products

The original dataset contains products from multiple categories on Nykaa,
including unrelated categories such as maternity, nutrition, and appliances.

For PrettyPlease, we create a focused beauty catalogue while preserving
the original dataset for reference.

In [19]:
beauty_categories = [
    "Makeup",
    "Natural",
    "Skin",
    "Personal Care",
    "Hair",
    "Fragrance",
    "Men's Store",
    "Nykaa Luxe",
    "Brand"
]

beauty_df = df[
    df["Product Category"]
    .str.split(">")
    .str[0]
    .str.strip()
    .isin(beauty_categories)
].copy()

print(f"Original products: {len(df)}")
print(f"Beauty products: {len(beauty_df)}")

Original products: 625
Beauty products: 481


In [20]:
beauty_df["Product Category"].str.split(">").str[0].str.strip().value_counts()

Product Category
Makeup           165
Natural           98
Skin              73
Personal Care     35
Brand             30
Nykaa Luxe        30
Hair              21
Men's Store       18
Fragrance         11
Name: count, dtype: int64

In [21]:
beauty_df.head().T

,0,1,2,4,5
Product Id,b77f3da33be6e65f6183da6ada8c07ca,f54658c5d511195b6621a640fb743b1b,cdf6b3387f8976c8e38ad150173dbb6d,f7f76573099db0058ef5264c35d9d02e,ee007f19f85ce1d73de2aa745ea1bc20
Product Brand Code,BZ1000,BH5931,BH7276,BZ1000,BZ1000
Retailer,nykaa.com,nykaa.com,nykaa.com,nykaa.com,nykaa.com
Product Category,Makeup > Face > Contour,Brand > L'Oreal Paris,Makeup > Face > Foundation,Makeup > Lips > Lip Stain,Natural > Shop By Concern > Acne Treatment
Product Brand,ETUDE HOUSE,L'Oreal Paris,The Body Shop,Nykaa Cosmetics,HealthVit
Product Name,ETUDE HOUSE Face Color Shading - 02,L'Oreal Paris Glow Mon Amour Highlighting Drop...,The Body Shop All-In-One Face Base - 045,Nykaa Wonderpuff Cushion Liquid Lipstick - Wer...,HealthVit Activated Charcoal Powder
Product Price,600,454,1395,539,249
Product Url,https://www.nykaa.com/c/p/760922?skuId=760920,https://www.nykaa.com/c/p/565974?skuId=565973,https://www.nykaa.com/c/p/31142?skuId=30050,https://www.nykaa.com/c/p/555315?skuId=555310,https://www.nykaa.com/c/p/137267?skuId=137271
Market,IN,IN,IN,IN,IN
Product Description,Etude House Face Color Shading provides a shad...,It's time to skip the snooze button and get up...,Note: The Body Shop products will be dispatche...,It's no secret that a good lippie is a real mo...,Experience the goodness of Pure Activated Char...


In [22]:
#checking if nykaa.com is still present in the beauty_df
beauty_df["Product Category"].eq("nykaa.com").sum()

0

### 6.3 Check Missing Categories

In [23]:
beauty_df["Product Category"].isna().sum()

0

### 6.4 Cleaning Numerical Features

Several numerical attributes are stored as strings in the raw dataset.
We convert price, rating, and review count into appropriate numerical
formats so they can be used for filtering and ranking.

In [24]:
numeric_columns = [
    "Product Price",
    "Product Rating",
    "Product Reviews Count"
]

for col in numeric_columns:
    beauty_df[col] = pd.to_numeric(beauty_df[col], errors="coerce")

In [25]:
beauty_df[numeric_columns].dtypes

Product Price              int64
Product Rating           float64
Product Reviews Count    float64
dtype: object

In [26]:
beauty_df[numeric_columns].isnull().sum()

Product Price             0
Product Rating           15
Product Reviews Count    15
dtype: int64

In [27]:
beauty_df[numeric_columns].describe()

,Product Price,Product Rating,Product Reviews Count
count,481.000000,466.000000,466.000000
mean,902.095634,4.241416,397.972103
std,1276.378620,0.474445,1109.641415
min,25.000000,1.000000,0.000000
25%,175.000000,4.000000,4.000000
50%,360.000000,4.300000,24.000000
75%,960.000000,4.500000,163.000000
max,7900.000000,5.000000,8513.000000


In [28]:
# checking the product text
beauty_df[
    [
        "Product Name",
        "Product Category",
        "Product Description",
        "Product Tags",
        "Product Contents"
    ]
].head(3).T

,0,1,2
Product Name,ETUDE HOUSE Face Color Shading - 02,L'Oreal Paris Glow Mon Amour Highlighting Drop...,The Body Shop All-In-One Face Base - 045
Product Category,Makeup > Face > Contour,Brand > L'Oreal Paris,Makeup > Face > Foundation
Product Description,Etude House Face Color Shading provides a shad...,It's time to skip the snooze button and get up...,Note: The Body Shop products will be dispatche...
Product Tags,"ETUDE HOUSE Face Color Shading - 02, Makeup, F...",L'Oreal Paris Glow Mon Amour Highlighting Drop...,"The Body Shop All-In-One Face Base - 045 , Mak..."
Product Contents,NaN,"G927637, Cyclopentasiloxane, Dimethicone, Isod...",NaN


### 6.5 Handling Missing Values

Product descriptions are available for all products, while ingredients,
ratings, and review counts may be missing for some products. Missing
textual attributes are replaced with empty strings, while missing
numerical attributes are retained as missing values and handled during
ranking.

In [29]:
text_columns = [
    "Product Name",
    "Product Category",
    "Product Description",
    "Product Tags",
    "Product Contents",
    "Product Brand"
]

for col in text_columns:
    beauty_df[col] = beauty_df[col].fillna("")

In [30]:
beauty_df[text_columns].isnull().sum()

Product Name           0
Product Category       0
Product Description    0
Product Tags           0
Product Contents       0
Product Brand          0
dtype: int64

### 6.6 Creating Product Representations
We want every product represented by the information that describes it.
To enable content-based recommendation, we combine the most informative
textual attributes of each product into a single product representation.

This representation will later be converted into embeddings and used to
measure semantic similarity between a user's preferences and available
products.

In [31]:
beauty_df["product_text"] = (
    beauty_df["Product Name"] + " " +
    beauty_df["Product Brand"] + " " +
    beauty_df["Product Category"] + " " +
    beauty_df["Product Tags"] + " " +
    beauty_df["Product Description"] + " " +
    beauty_df["Product Contents"]
)

In [32]:
beauty_df[["Product Name", "product_text"]].head(3)

,Product Name,product_text
0,ETUDE HOUSE Face Color Shading - 02,ETUDE HOUSE Face Color Shading - 02 ETUDE HOUS...
1,L'Oreal Paris Glow Mon Amour Highlighting Drop...,L'Oreal Paris Glow Mon Amour Highlighting Drop...
2,The Body Shop All-In-One Face Base - 045,The Body Shop All-In-One Face Base - 045 The B...


## 7. Feature Engineering

### 7.1 Hierarchical Category Features

The product category contains hierarchical information. We split it into
individual levels so that product type, product domain, and broader
categories can be used independently during recommendation.

In [33]:
category_parts = beauty_df["Product Category"].str.split(">")

In [34]:
category_split = beauty_df["Product Category"].str.split(">", expand=True)

beauty_df["category_level_1"] = category_split[0].str.strip()
beauty_df["category_level_2"] = category_split[1].str.strip()
beauty_df["category_level_3"] = category_split[2].str.strip()

In [35]:
beauty_df[
    [
        "Product Category",
        "category_level_1",
        "category_level_2",
        "category_level_3"
    ]
].head(10)

,Product Category,category_level_1,category_level_2,category_level_3
0,Makeup > Face > Contour,Makeup,Face,Contour
1,Brand > L'Oreal Paris,Brand,L'Oreal Paris,None
2,Makeup > Face > Foundation,Makeup,Face,Foundation
4,Makeup > Lips > Lip Stain,Makeup,Lips,Lip Stain
5,Natural > Shop By Concern > Acne Treatment,Natural,Shop By Concern,Acne Treatment
6,Brand > Nivea,Brand,Nivea,None
7,Natural > Skin > Face Wash,Natural,Skin,Face Wash
11,Makeup > Face > Concealer,Makeup,Face,Concealer
12,Personal Care > Face > Moisturizer,Personal Care,Face,Moisturizer
13,Skin > Body Care > Lotions & Creams,Skin,Body Care,Lotions & Creams


In [36]:
# checking the levels
beauty_df["category_level_1"].value_counts()

category_level_1
Makeup           165
Natural           98
Skin              73
Personal Care     35
Brand             30
Nykaa Luxe        30
Hair              21
Men's Store       18
Fragrance         11
Name: count, dtype: int64

In [37]:
beauty_df["category_level_2"].value_counts().head(20)

category_level_2
Face               68
Shop By Concern    57
Lips               48
Types of Skin      28
Eyes               26
Body Care          22
Moisturizers       18
Hair               17
Skin               15
Makeup             15
Bath & Shower      14
Nails              13
Fragrance          10
Makeup Kits        10
Hair Care           9
Combos @ Nykaa      8
Gifts @ Nykaa       8
Himalaya            7
Lakme               7
Cleansers           7
Name: count, dtype: int64

In [38]:
beauty_df["category_level_3"].value_counts().head(20)

category_level_3
Lipstick                        25
Blush                           15
Shampoo                         14
Liquid Lipstick                 14
Nail Polish                     13
Lotions & Creams                13
Foundation                      12
Face Moisturizer & Day Cream    12
Face                            10
Under Eye Concealer             10
Conditioner                      9
Skin Dryness                     9
Highlighters                     9
Normal Skin                      8
Shampoo & Cleanser               8
Dry Skin                         8
Concealer                        8
Perfumes (EDT & EDP)             7
Skin Brightening                 7
Perfumes (EDP/EDT)               7
Name: count, dtype: int64

In [39]:
# we don't want the recommender dealing with "Makeup > Face > Foundation".
df.columns.tolist()

['Product Id',
 'Product Brand Code',
 'Retailer',
 'Product Category',
 'Product Brand',
 'Product Name',
 'Product Price',
 'Product Url',
 'Market',
 'Product Description',
 'Product Currency',
 'Product Image Url',
 'Product Tags',
 'Product Contents',
 'Product Rating',
 'Product Reviews Count',
 'Expected Category Count',
 'Expected Brand Count']

In [40]:
beauty_df["product_type"] = beauty_df["category_level_3"].fillna(
    beauty_df["category_level_2"]
)

In [41]:
beauty_df["product_type"].value_counts().head(20)

product_type
Lipstick                        25
Blush                           15
Shampoo                         14
Liquid Lipstick                 14
Nail Polish                     13
Lotions & Creams                13
Foundation                      12
Face Moisturizer & Day Cream    12
Face                            10
Under Eye Concealer             10
Skin Dryness                     9
Highlighters                     9
Combos @ Nykaa                   9
Conditioner                      9
Gifts @ Nykaa                    8
Normal Skin                      8
Dry Skin                         8
Concealer                        8
Shampoo & Cleanser               8
Perfumes (EDT & EDP)             7
Name: count, dtype: int64

In [42]:
# create a clean brand

beauty_df["brand"] = (
    beauty_df["Product Brand"]
    .str.strip()
    .str.lower()
)

In [43]:
# create price bands

beauty_df["Product Price"] = pd.to_numeric(
    beauty_df["Product Price"], errors="coerce"
)

beauty_df["price_band"] = pd.cut(
    beauty_df["Product Price"],
    bins=[0, 300, 700, 1500, float("inf")],
    labels=["Budget", "Mid-range", "Premium", "Luxury"]
)

In [44]:
# create a quality score
import numpy as np

beauty_df["Product Rating"] = pd.to_numeric(
    beauty_df["Product Rating"], errors="coerce"
)

beauty_df["Product Reviews Count"] = pd.to_numeric(
    beauty_df["Product Reviews Count"], errors="coerce"
)

beauty_df["quality_score"] = (
    beauty_df["Product Rating"] * 
    np.log1p(beauty_df["Product Reviews Count"])
)

This gives us something like:

4.5★ with 2,000 reviews → stronger signal

5.0★ with 1 review → weaker signal

### 7.2 Check the Dataset

In [45]:
beauty_df[
    [
        "Product Name",
        "product_type",
        "brand",
        "Product Price",
        "price_band",
        "Product Rating",
        "Product Reviews Count",
        "quality_score"
    ]
].head(10)

,Product Name,product_type,brand,Product Price,price_band,Product Rating,Product Reviews Count,quality_score
0,ETUDE HOUSE Face Color Shading - 02,Contour,etude house,600,Mid-range,4.6,4.0,7.403414
1,L'Oreal Paris Glow Mon Amour Highlighting Drop...,L'Oreal Paris,l'oreal paris,454,Mid-range,4.3,147.0,21.488013
2,The Body Shop All-In-One Face Base - 045,Foundation,the body shop,1395,Premium,4.4,58.0,17.941165
4,Nykaa Wonderpuff Cushion Liquid Lipstick - Wer...,Lip Stain,nykaa cosmetics,539,Mid-range,4.0,934.0,27.362186
5,HealthVit Activated Charcoal Powder,Acne Treatment,healthvit,249,Budget,4.2,442.0,25.592993
6,NIVEA Body Lotion Oil in Lotion Rose & Argan O...,Nivea,nivea,260,Budget,4.5,1220.0,31.983415
7,Lotus Herbals Whiteglow Activated Charcoal Bri...,Face Wash,lotus herbals,140,Budget,4.2,8.0,9.228343
11,Smashbox Color Correcting Stick - Look Less Ti...,Concealer,smashbox,2090,Luxury,4.1,13.0,10.820135
12,NIVEA Creme - All Season Multi Purpose Cream,Moisturizer,nivea,175,Budget,4.5,1177.0,31.822080
13,Vaseline Ice Cool Hydration Lotion,Lotions & Creams,vaseline,210,Budget,4.3,136.0,21.155918


## 8. Recommendation Text

In [46]:
beauty_df["recommendation_text"] = (
    beauty_df["Product Name"].fillna("") + " " +
    beauty_df["product_type"].fillna("") + " " +
    beauty_df["brand"].fillna("") + " " +
    beauty_df["Product Description"].fillna("") + " " +
    beauty_df["Product Tags"].fillna("")
)

In [47]:
beauty_df[["Product Name", "recommendation_text"]].head(3)

,Product Name,recommendation_text
0,ETUDE HOUSE Face Color Shading - 02,ETUDE HOUSE Face Color Shading - 02 Contour et...
1,L'Oreal Paris Glow Mon Amour Highlighting Drop...,L'Oreal Paris Glow Mon Amour Highlighting Drop...
2,The Body Shop All-In-One Face Base - 045,The Body Shop All-In-One Face Base - 045 Found...


TF-IDF + Cosine Similarity initially, will be improving with semantic embeddings and ranking signals later!!

### 8.1 Convert Products → TF-IDF Vectors
Now we get to ML :)

In [48]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

tfidf_matrix = vectorizer.fit_transform(
    beauty_df["recommendation_text"]
)

tfidf_matrix.shape

(481, 5000)

In [49]:
# calculate cosine similarity
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(tfidf_matrix)

similarity_matrix.shape

(481, 481)

### 8.2 Building a basic recommender

In [50]:
def recommend_products(product_name, n=5):
    idx = beauty_df[
        beauty_df["Product Name"].str.contains(
            product_name,
            case=False,
            na=False
        )
    ].index[0]

    scores = similarity_matrix[idx]

    similar_indices = scores.argsort()[::-1][1:n+1]

    recommendations = beauty_df.iloc[similar_indices][[
        "Product Name",
        "Product Brand",
        "Product Price",
        "Product Rating",
        "product_type"
    ]].copy()

    recommendations["similarity"] = scores[similar_indices]

    return recommendations

In [51]:
recommend_products("Face Color Shading")

,Product Name,Product Brand,Product Price,Product Rating,product_type,similarity
434,L.A. Colors Highlight & Contour Palette - Ligh...,L.A. Colors,325,3.8,Contour,0.207571
302,House Of Makeup Double Duty Kohl + Liner - Sil...,House of Makeup,499,2.8,Kajal,0.184576
154,Makeup Revolution HD Pro Ultra Powder Contour ...,Makeup Revolution,1850,4.2,Highlighters,0.167644
256,L.A. Girl Pro Contour Cream - Deep,L.A. Girl,675,4.0,Highlighters,0.160931
34,Provoc Contour Correct Conceal Palette - Profi...,Provoc,1850,3.9,Face Palettes,0.094530


In [52]:
# We now use the TF-IDF cosine similarity matrix to retrieve products that are most similar to a selected product.

def recommend_products(product_name, top_n=5):
    # Find the product
    matches = beauty_df[
        beauty_df["Product Name"].str.contains(
            product_name, case=False, na=False
        )
    ]

    if matches.empty:
        return "Product not found."

    idx = matches.index[0]

    # Get similarity scores
    similarity_scores = similarity_matrix[idx]

    # Sort by similarity
    similar_indices = similarity_scores.argsort()[::-1]

    # Remove the selected product itself
    similar_indices = [
        i for i in similar_indices
        if i != idx
    ]

    # Select top recommendations
    top_indices = similar_indices[:top_n]

    recommendations = beauty_df.loc[
        beauty_df.index[top_indices],
        [
            "Product Name",
            "Product Brand",
            "Product Price",
            "Product Rating",
            "product_type",
            "price_band",
            "quality_score"
        ]
    ].copy()

    recommendations["similarity"] = [
        similarity_scores[i] for i in top_indices
    ]

    return recommendations

In [53]:
recommend_products(
    "ETUDE HOUSE Face Color Shading - 02",
    top_n=5
)

,Product Name,Product Brand,Product Price,Product Rating,product_type,price_band,quality_score,similarity
434,L.A. Colors Highlight & Contour Palette - Ligh...,L.A. Colors,325,3.8,Contour,Mid-range,15.494642,0.207571
302,House Of Makeup Double Duty Kohl + Liner - Sil...,House of Makeup,499,2.8,Kajal,Mid-range,3.881624,0.184576
154,Makeup Revolution HD Pro Ultra Powder Contour ...,Makeup Revolution,1850,4.2,Highlighters,Luxury,13.519278,0.167644
256,L.A. Girl Pro Contour Cream - Deep,L.A. Girl,675,4.0,Highlighters,Mid-range,18.460482,0.160931
34,Provoc Contour Correct Conceal Palette - Profi...,Provoc,1850,3.9,Face Palettes,Luxury,10.292324,0.094530


TF-IDF is capturing textual similarity, but not necessarily product similarity.

### 8.3 Let's build a better, hybrid recommender

Final Score =
    70% Text Similarity
  + 15% Product Type Match
  + 10% Price Band Match
  + 5% Brand Match

In [54]:
beauty_df = beauty_df.reset_index(drop=True)

In [55]:
print(beauty_df.shape)
print(beauty_df.index[:10])
print(similarity_matrix.shape)

(481, 27)
RangeIndex(start=0, stop=10, step=1)
(481, 481)


In [56]:
def hybrid_recommend_products(product_name, top_n=5):
    
    # Find the product
    matches = beauty_df[
        beauty_df["Product Name"].str.contains(
            product_name, case=False, na=False
        )
    ]

    if matches.empty:
        return "Product not found."

    idx = matches.index[0]
    
    # Get the selected product
    target = beauty_df.loc[idx]

    # Text similarity
    text_scores = similarity_matrix[idx]

    # Product type match
    type_scores = (
        beauty_df["product_type"] == target["product_type"]
    ).astype(float)

    # Price band match
    price_scores = (
        beauty_df["price_band"] == target["price_band"]
    ).astype(float)

    # Brand match
    brand_scores = (
        beauty_df["brand"] == target["brand"]
    ).astype(float)

    # Final weighted score
    final_scores = (
        0.70 * text_scores +
        0.15 * type_scores +
        0.10 * price_scores +
        0.05 * brand_scores
    )

    # Sort recommendations
    similar_indices = final_scores.argsort()[::-1]

    # Remove the product itself
    similar_indices = [
        i for i in similar_indices
        if i != idx
    ]

    # Top N
    top_indices = similar_indices[:top_n]

    recommendations = beauty_df.loc[
        beauty_df.index[top_indices],
        [
            "Product Name",
            "Product Brand",
            "Product Price",
            "Product Rating",
            "product_type",
            "price_band",
            "quality_score"
        ]
    ].copy()

    recommendations["text_similarity"] = [
        text_scores[i] for i in top_indices
    ]

    recommendations["final_score"] = [
        final_scores[i] for i in top_indices
    ]

    return recommendations

In [57]:
hybrid_recommend_products(
    "ETUDE HOUSE Face Color Shading - 02",
    top_n=5
)

,Product Name,Product Brand,Product Price,Product Rating,product_type,price_band,quality_score,text_similarity,final_score
325,L.A. Colors Highlight & Contour Palette - Ligh...,L.A. Colors,325,3.8,Contour,Mid-range,15.494642,0.207571,0.395300
222,House Of Makeup Double Duty Kohl + Liner - Sil...,House of Makeup,499,2.8,Kajal,Mid-range,3.881624,0.184576,0.229203
187,L.A. Girl Pro Contour Cream - Deep,L.A. Girl,675,4.0,Highlighters,Mid-range,18.460482,0.160931,0.212652
446,Nykaa Get Cheeky! Blush Duo Palette - Cali Chi...,Nykaa Cosmetics,629,4.4,Combos @ Nykaa,Mid-range,30.765224,0.068462,0.147923
176,Nykaa Get Cheeky! Blush Duo Palette - Cali Chi...,Nykaa Cosmetics,629,4.4,Blush,Mid-range,30.765224,0.065989,0.146192


### 8.4 Duplicate Product Analysis

Before evaluating recommendation quality, we check for duplicate products to ensure that the recommendation engine does not return multiple records representing the same product.

In [58]:
duplicate_products = beauty_df[
    beauty_df["Product Id"].duplicated(keep=False)
].sort_values("Product Id")

print("Duplicate product records:", len(duplicate_products))

display(
    duplicate_products[
        ["Product Id", "Product Name", "Product Brand", "product_type"]
    ]
)

Duplicate product records: 74


,Product Id,Product Name,Product Brand,product_type
238,017044830af4f3456d17c571f46a032a,Lime Crime Wet Cherry Lip Gloss - Cherry Candy,Lime Crime,Lip Gloss
130,017044830af4f3456d17c571f46a032a,Lime Crime Wet Cherry Lip Gloss - Cherry Candy,Lime Crime,Lip Gloss
275,0261186cfbfe3041c88fbab962ce6625,Richfeel Hair Conditioner,Richfeel,Conditioner
399,0261186cfbfe3041c88fbab962ce6625,Richfeel Hair Conditioner,Richfeel,Conditioner
262,15496cfbe7ee2e663d978725a25764d6,L'Occitane Immortelle Divine Cream,L'Occitane,Anti-Ageing
...,...,...,...,...
113,ec96ed3a148fcfbf7e62a85777f09427,Lotus Herbals Jojobawash Active Milli Capsules...,Lotus Herbals,Lotus Herbals
229,f7f76573099db0058ef5264c35d9d02e,Nykaa Wonderpuff Cushion Liquid Lipstick - Wer...,Nykaa Cosmetics,Lip Stain
3,f7f76573099db0058ef5264c35d9d02e,Nykaa Wonderpuff Cushion Liquid Lipstick - Wer...,Nykaa Cosmetics,Lip Stain
381,fbe59721a5e29ad76fc386eb1e4270ef,Biotique Bio Papaya Revitalizing Tan removal S...,Biotique,Scrubs & Exfoliators


In [59]:
duplicate_names = beauty_df[
    beauty_df["Product Name"].duplicated(keep=False)
].sort_values("Product Name")

print("Duplicate product names:", len(duplicate_names))

display(
    duplicate_names[
        ["Product Name", "Product Brand", "product_type"]
    ]
)

Duplicate product names: 59


,Product Name,Product Brand,product_type
430,Aveda Dry Remedy Moisturizing Conditioner,Aveda,Conditioner
257,Aveda Dry Remedy Moisturizing Conditioner,Aveda,Dry & Frizzy Hair
309,Biotique Bio Kelp Protein Shampoo For Falling ...,Biotique,Shampoo & Cleanser
85,Biotique Bio Kelp Protein Shampoo For Falling ...,Biotique,Peppermint Oil
381,Biotique Bio Papaya Revitalizing Tan removal S...,Biotique,Scrubs & Exfoliators
29,Biotique Bio Papaya Revitalizing Tan removal S...,Biotique,Biotique
397,Burberry My Burberry Eau De Toilette,Burberry,Perfumes (EDP/EDT)
21,Burberry My Burberry Eau De Toilette,Burberry,Perfumes (EDT & EDP)
209,Charlotte Tilbury Luxury Palette - The Sophist...,Charlotte Tilbury,Palettes
60,Charlotte Tilbury Luxury Palette - The Sophist...,Charlotte Tilbury,Combos @ Nykaa


In [60]:
print("Total rows:", len(beauty_df))
print("Unique Product IDs:", beauty_df["Product Id"].nunique())
print("Duplicate Product IDs:", beauty_df["Product Id"].duplicated().sum())

Total rows: 481
Unique Product IDs: 437
Duplicate Product IDs: 44


In [61]:
duplicate_products = beauty_df[
    beauty_df["Product Id"].duplicated(keep=False)
].sort_values("Product Id")

duplicate_products[
    ["Product Id", "Product Name", "Product Brand", "product_type"]
].head(100)

,Product Id,Product Name,Product Brand,product_type
238,017044830af4f3456d17c571f46a032a,Lime Crime Wet Cherry Lip Gloss - Cherry Candy,Lime Crime,Lip Gloss
130,017044830af4f3456d17c571f46a032a,Lime Crime Wet Cherry Lip Gloss - Cherry Candy,Lime Crime,Lip Gloss
275,0261186cfbfe3041c88fbab962ce6625,Richfeel Hair Conditioner,Richfeel,Conditioner
399,0261186cfbfe3041c88fbab962ce6625,Richfeel Hair Conditioner,Richfeel,Conditioner
262,15496cfbe7ee2e663d978725a25764d6,L'Occitane Immortelle Divine Cream,L'Occitane,Anti-Ageing
...,...,...,...,...
113,ec96ed3a148fcfbf7e62a85777f09427,Lotus Herbals Jojobawash Active Milli Capsules...,Lotus Herbals,Lotus Herbals
229,f7f76573099db0058ef5264c35d9d02e,Nykaa Wonderpuff Cushion Liquid Lipstick - Wer...,Nykaa Cosmetics,Lip Stain
3,f7f76573099db0058ef5264c35d9d02e,Nykaa Wonderpuff Cushion Liquid Lipstick - Wer...,Nykaa Cosmetics,Lip Stain
381,fbe59721a5e29ad76fc386eb1e4270ef,Biotique Bio Papaya Revitalizing Tan removal S...,Biotique,Scrubs & Exfoliators


In [62]:
type_counts = (
    beauty_df.groupby("Product Id")["product_type"]
    .nunique()
)

print("Products with multiple types:", (type_counts > 1).sum())
print("Products with duplicate identical types:", (type_counts == 1).sum())

Products with multiple types: 23
Products with duplicate identical types: 414


In [63]:
beauty_df = (
    beauty_df
    .groupby("Product Id", as_index=False)
    .agg({
        "Product Brand Code": "first",
        "Retailer": "first",
        "Product Category": lambda x: " | ".join(x.dropna().unique()),
        "Product Brand": "first",
        "Product Name": "first",
        "Product Price": "first",
        "Product Url": "first",
        "Market": "first",
        "Product Description": "first",
        "Product Currency": "first",
        "Product Image Url": "first",
        "Product Tags": "first",
        "Product Contents": "first",
        "Product Rating": "first",
        "Product Reviews Count": "first",
        "Expected Category Count": "first",
        "Expected Brand Count": "first",
        "product_text": "first",
        "category_level_1": lambda x: " | ".join(x.dropna().unique()),
        "category_level_2": lambda x: " | ".join(x.dropna().unique()),
        "category_level_3": lambda x: " | ".join(x.dropna().unique()),
        "product_type": lambda x: " | ".join(x.dropna().unique()),
        "brand": "first",
        "price_band": "first",
        "quality_score": "first",
        "recommendation_text": "first"
    })
)

In [64]:
print(beauty_df.shape)
print(beauty_df["Product Id"].nunique())
print(beauty_df["Product Id"].duplicated().sum())

(437, 27)
437
0


If a product appeared twice with the same category, keep it once.
If it appeared with different categories, combine them.

In [65]:
beauty_df["product_text"] = (
    beauty_df["Product Name"].fillna("") + " " +
    beauty_df["Product Brand"].fillna("") + " " +
    beauty_df["product_type"].fillna("") + " " +
    beauty_df["Product Description"].fillna("") + " " +
    beauty_df["Product Tags"].fillna("") + " " +
    beauty_df["Product Contents"].fillna("")
)

In [66]:
beauty_df[["Product Name", "product_text"]].head(3)

,Product Name,product_text
0,Maybelline New York Baby Lips Color Candy Rush...,Maybelline New York Baby Lips Color Candy Rush...
1,Lime Crime Wet Cherry Lip Gloss - Cherry Candy,Lime Crime Wet Cherry Lip Gloss - Cherry Candy...
2,VLCC Ayurveda Baby Cream,VLCC Ayurveda Baby Cream VLCC Skin Dryness Enr...


In [67]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=1
)

tfidf_matrix = tfidf.fit_transform(beauty_df["product_text"])

print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (437, 56389)


In [68]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(tfidf_matrix)

print("Similarity matrix shape:", similarity_matrix.shape)

Similarity matrix shape: (437, 437)


In [69]:
def recommend_products(product_name, top_n=5):
    # Find the product
    matches = beauty_df[
        beauty_df["Product Name"].str.lower() == product_name.lower()
    ]

    if matches.empty:
        return "Product not found"

    idx = matches.index[0]

    # Get similarity scores
    scores = similarity_matrix[idx]

    # Get highest-scoring products
    similar_indices = scores.argsort()[::-1]

    # Remove the product itself
    similar_indices = [
        i for i in similar_indices
        if i != idx
    ]

    # Take top N
    top_indices = similar_indices[:top_n]

    recommendations = beauty_df.loc[
        top_indices,
        [
            "Product Name",
            "Product Brand",
            "Product Price",
            "Product Rating",
            "product_type"
        ]
    ].copy()

    recommendations["similarity"] = [
        scores[i] for i in top_indices
    ]

    return recommendations

In [70]:
recommend_products(
    "ETUDE HOUSE Face Color Shading - 02",
    top_n=10
)

,Product Name,Product Brand,Product Price,Product Rating,product_type,similarity
193,L.A. Colors Highlight & Contour Palette - Ligh...,L.A. Colors,325,3.8,Contour,0.095452
96,L.A. Girl Pro Contour Cream - Deep,L.A. Girl,675,4.0,Highlighters,0.085879
176,Makeup Revolution HD Pro Ultra Powder Contour ...,Makeup Revolution,1850,4.2,Highlighters,0.083160
349,House Of Makeup Double Duty Kohl + Liner - Sil...,House of Makeup,499,2.8,Kajal,0.081206
234,Provoc Contour Correct Conceal Palette - Profi...,Provoc,1850,3.9,Face Palettes,0.050030
76,Nykaa Get Cheeky! Blush Duo Palette - Cali Chi...,Nykaa Cosmetics,629,4.4,Blush | Combos @ Nykaa,0.037309
138,The Face Shop Club Ryan Velvet Lip Tint - Grac...,The Face Shop,793,4.5,Liquid Lipstick,0.034249
399,Smashbox L.A. Lights Blendable Lip & Cheek Col...,Smashbox,2300,4.1,Blush,0.031465
33,Lotus Make-Up Pure Colors Matte Lip Color - 59...,Lotus Make Up,236,4.1,Lipstick,0.031312
106,Sulwhasoo First Care Activating Serum,Sulwhasoo,1665,4.5,Anti-Ageing,0.030100


In [71]:
# Adding a new column quality_score_norm scaled 0-1, so popularity can't drown out relevance later.
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
beauty_df["quality_score_norm"] = scaler.fit_transform(beauty_df[["quality_score"]].fillna(0))

beauty_df[["Product Name", "quality_score", "quality_score_norm"]].sort_values("quality_score_norm", ascending=False).head()

,Product Name,quality_score,quality_score_norm
427,Biotique Bio Papaya Revitalizing Tan removal S...,39.817655,1.000000
277,Ponds Super Light Gel Oil Free Moisturiser Wit...,38.695309,0.971813
155,Nykaa Naturals Citronella Essential Oil,37.466195,0.940944
47,Clinique Moisture Surge 72-Hour Auto-Replenish...,36.772973,0.923534
116,TONYMOLY I'M Red Wine Mask Sheet,36.751807,0.923003


In [72]:
def type_set(value):
    if not isinstance(value, str) or not value.strip():
        return set()
    return {t.strip().lower() for t in value.split("|") if t.strip()}

def type_overlap_score(target_set, candidate_set):
    if not target_set or not candidate_set:
        return 0.0
    return len(target_set & candidate_set) / len(target_set | candidate_set)

beauty_df["product_type_set"] = beauty_df["product_type"].apply(type_set)

In [73]:
# new one

def category_match_score(target_row, candidate_row):
    # category_level_2 is broader than product_type, e.g. "Face", "Eyes", "Lips"
    t2 = str(target_row["category_level_2"]).strip().lower()
    c2 = str(candidate_row["category_level_2"]).strip().lower()
    if t2 and c2 and t2 == c2:
        return 1.0
    return 0.0

def hybrid_recommend_v3(product_name, top_n=10, weights=None):
    if weights is None:
        weights = {"text": 0.45, "type": 0.20, "category2": 0.20, "price": 0.10, "quality": 0.05}

    matches = beauty_df[beauty_df["Product Name"].str.contains(product_name, case=False, na=False)]
    if matches.empty:
        return "Product not found."

    idx = matches.index[0]
    target = beauty_df.loc[idx]

    text_scores = similarity_matrix[idx]
    target_types = target["product_type_set"]
    type_scores = beauty_df["product_type_set"].apply(lambda s: type_overlap_score(target_types, s)).to_numpy()
    category2_scores = beauty_df.apply(lambda row: category_match_score(target, row), axis=1).to_numpy()
    price_scores = (beauty_df["price_band"] == target["price_band"]).astype(float).to_numpy()
    quality_scores = beauty_df["quality_score_norm"].fillna(0).to_numpy()

    final_scores = (
        weights["text"] * text_scores +
        weights["type"] * type_scores +
        weights["category2"] * category2_scores +
        weights["price"] * price_scores +
        weights["quality"] * quality_scores
    )

    order = final_scores.argsort()[::-1]
    order = [i for i in order if i != idx][:top_n]

    recommendations = beauty_df.loc[order, [
        "Product Name", "Product Brand", "Product Price", "Product Rating",
        "product_type", "category_level_2", "price_band"
    ]].copy()

    recommendations["text_similarity"] = text_scores[order]
    recommendations["type_overlap"] = type_scores[order]
    recommendations["category2_match"] = category2_scores[order]
    recommendations["final_score"] = final_scores[order]

    return recommendations.reset_index(drop=True)

hybrid_recommend_v3("ETUDE HOUSE Face Color Shading - 02", top_n=5)

,Product Name,Product Brand,Product Price,Product Rating,product_type,category_level_2,price_band,text_similarity,type_overlap,category2_match,final_score
0,L.A. Colors Highlight & Contour Palette - Ligh...,L.A. Colors,325,3.8,Contour,Face,Mid-range,0.095452,1.0,1.0,0.562411
1,L.A. Girl Pro Contour Cream - Deep,L.A. Girl,675,4.0,Highlighters,Face,Mid-range,0.085879,0.0,1.0,0.361827
2,Wet n Wild Color Icon Blush - Fantastic Plasti...,Wet n Wild,399,4.0,Blush,Face,Mid-range,0.024360,0.0,1.0,0.345749
3,Nykaa SKINgenius Skin Perfecting & Hydrating M...,Nykaa Cosmetics,368,4.1,Highlighters,Face,Mid-range,0.016023,0.0,1.0,0.341105
4,Faces Canada Glam On Perfect Blush - Apricot 06,Faces Canada,503,4.2,Blush,Face,Mid-range,0.017502,0.0,1.0,0.338920


In [74]:
def hybrid_recommend_v4(product_name, top_n=10, weights=None):
    if weights is None:
        weights = {"text": 0.45, "type": 0.20, "category2": 0.20, "price": 0.10, "quality": 0.05}

    matches = beauty_df[beauty_df["Product Name"].str.contains(product_name, case=False, na=False)]
    if matches.empty:
        return "Product not found."

    idx = matches.index[0]
    target = beauty_df.loc[idx]

    # NEW: only consider candidates in the same top-level category (Makeup vs Skin vs Hair etc.)
    same_l1 = beauty_df["category_level_1"].astype(str).str.strip().str.lower() == str(target["category_level_1"]).strip().lower()
    pool = beauty_df[same_l1].copy()
    pool_positions = pool.index.to_numpy()

    text_scores = similarity_matrix[idx][pool_positions]
    target_types = target["product_type_set"]
    type_scores = pool["product_type_set"].apply(lambda s: type_overlap_score(target_types, s)).to_numpy()
    category2_scores = pool.apply(lambda row: category_match_score(target, row), axis=1).to_numpy()
    price_scores = (pool["price_band"] == target["price_band"]).astype(float).to_numpy()
    quality_scores = pool["quality_score_norm"].fillna(0).to_numpy()

    final_scores = (
        weights["text"] * text_scores +
        weights["type"] * type_scores +
        weights["category2"] * category2_scores +
        weights["price"] * price_scores +
        weights["quality"] * quality_scores
    )

    pool = pool.assign(final_score=final_scores, text_similarity=text_scores, type_overlap=type_scores, category2_match=category2_scores)
    pool = pool[pool.index != idx].sort_values("final_score", ascending=False).head(top_n)

    return pool[[
        "Product Name", "Product Brand", "Product Price", "Product Rating",
        "product_type", "category_level_1", "category_level_2", "price_band",
        "text_similarity", "type_overlap", "category2_match", "final_score"
    ]].reset_index(drop=True)

hybrid_recommend_v4("ETUDE HOUSE Face Color Shading - 02", top_n=5)

,Product Name,Product Brand,Product Price,Product Rating,product_type,category_level_1,category_level_2,price_band,text_similarity,type_overlap,category2_match,final_score
0,L.A. Colors Highlight & Contour Palette - Ligh...,L.A. Colors,325,3.8,Contour,Makeup,Face,Mid-range,0.095452,1.0,1.0,0.562411
1,L.A. Girl Pro Contour Cream - Deep,L.A. Girl,675,4.0,Highlighters,Makeup,Face,Mid-range,0.085879,0.0,1.0,0.361827
2,Wet n Wild Color Icon Blush - Fantastic Plasti...,Wet n Wild,399,4.0,Blush,Makeup,Face,Mid-range,0.024360,0.0,1.0,0.345749
3,Nykaa SKINgenius Skin Perfecting & Hydrating M...,Nykaa Cosmetics,368,4.1,Highlighters,Makeup,Face,Mid-range,0.016023,0.0,1.0,0.341105
4,Faces Canada Glam On Perfect Blush - Apricot 06,Faces Canada,503,4.2,Blush,Makeup,Face,Mid-range,0.017502,0.0,1.0,0.338920


In [75]:
# HealthVit Activated Charcoal Powder

def hybrid_recommend_v4(product_name, top_n=10, weights=None):
    if weights is None:
        weights = {"text": 0.45, "type": 0.20, "category2": 0.20, "price": 0.10, "quality": 0.05}

    matches = beauty_df[beauty_df["Product Name"].str.contains(product_name, case=False, na=False)]
    if matches.empty:
        return "Product not found."

    idx = matches.index[0]
    target = beauty_df.loc[idx]

    # NEW: only consider candidates in the same top-level category (Makeup vs Skin vs Hair etc.)
    same_l1 = beauty_df["category_level_1"].astype(str).str.strip().str.lower() == str(target["category_level_1"]).strip().lower()
    pool = beauty_df[same_l1].copy()
    pool_positions = pool.index.to_numpy()

    text_scores = similarity_matrix[idx][pool_positions]
    target_types = target["product_type_set"]
    type_scores = pool["product_type_set"].apply(lambda s: type_overlap_score(target_types, s)).to_numpy()
    category2_scores = pool.apply(lambda row: category_match_score(target, row), axis=1).to_numpy()
    price_scores = (pool["price_band"] == target["price_band"]).astype(float).to_numpy()
    quality_scores = pool["quality_score_norm"].fillna(0).to_numpy()

    final_scores = (
            weights["text"] * text_scores +
            weights["type"] * type_scores +
            weights["category2"] * category2_scores +
            weights["price"] * price_scores +
            weights["quality"] * quality_scores
        )
    
    pool = pool.assign(final_score=final_scores, text_similarity=text_scores, type_overlap=type_scores, category2_match=category2_scores)
    pool = pool[pool.index != idx].sort_values("final_score", ascending=False).head(top_n)
    
    return pool[[
            "Product Name", "Product Brand", "Product Price", "Product Rating",
            "product_type", "category_level_1", "category_level_2", "price_band",
            "text_similarity", "type_overlap", "category2_match", "final_score"
        ]].reset_index(drop=True)
    
hybrid_recommend_v4("HealthVit Activated Charcoal Powder", top_n=5) 

,Product Name,Product Brand,Product Price,Product Rating,product_type,category_level_1,category_level_2,price_band,text_similarity,type_overlap,category2_match,final_score
0,Aroma Treasures Tea Tree Face Wash,Aroma Treasures,240,4.1,Acne Treatment,Natural,Shop By Concern,Budget,0.024952,1.0,1.0,0.531266
1,Ancient Living Tea Tree Face Wash,Ancient Living,199,4.0,Acne Treatment,Natural,Shop By Concern,Budget,0.032853,1.0,1.0,0.521747
2,Biotique Bio Bhringraj Therapeutic Oil For Fal...,Biotique,159,4.2,Hairfall,Natural,Shop By Concern,Budget,0.016533,0.0,1.0,0.348326
3,Biotique Bio Honey Gel Refreshing Foaming Face...,Biotique,65,4.3,Pigmentation,Natural,Shop By Concern,Budget,0.020651,0.0,1.0,0.347214
4,Biotique Bio Morning Nectar Visibly Flawless F...,Biotique,49,4.1,Pigmentation,Natural,Shop By Concern,Budget,0.033038,0.0,1.0,0.333859


In [ ]:
def hybrid_recommend_v5(product_name, top_n=10, weights=None, min_text_similarity=0.03):
    if weights is None:
        weights = {"text": 0.45, "type": 0.20, "category2": 0.20, "price": 0.10, "quality": 0.05}

    matches = beauty_df[beauty_df["Product Name"].str.contains(product_name, case=False, na=False)]
    if matches.empty:
        return "Product not found."

    idx = matches.index[0]
    target = beauty_df.loc[idx]

    same_l1 = beauty_df["category_level_1"].astype(str).str.strip().str.lower() == str(target["category_level_1"]).strip().lower()
    pool = beauty_df[same_l1].copy()
    pool_positions = pool.index.to_numpy()

    text_scores = similarity_matrix[idx][pool_positions]

    # NEW: drop candidates with essentially no textual relationship to the target,
    # regardless of how well their category/type matches
    keep = text_scores >= min_text_similarity
    pool = pool[keep]
    if pool.empty:
        return f"No sufficiently similar products found for '{product_name}'. The catalog may not have close matches in this category."
    text_scores = text_scores[keep]

    target_types = target["product_type_set"]
    type_scores = pool["product_type_set"].apply(lambda s: type_overlap_score(target_types, s)).to_numpy()
    category2_scores = pool.apply(lambda row: category_match_score(target, row), axis=1).to_numpy()
    price_scores = (pool["price_band"] == target["price_band"]).astype(float).to_numpy()
    quality_scores = pool["quality_score_norm"].fillna(0).to_numpy()

    final_scores = (
        weights["text"] * text_scores +
        weights["type"] * type_scores +
        weights["category2"] * category2_scores +
        weights["price"] * price_scores +
        weights["quality"] * quality_scores
    )

    pool = pool.assign(final_score=final_scores, text_similarity=text_scores, type_overlap=type_scores, category2_match=category2_scores)
    pool = pool[pool.index != idx].sort_values("final_score", ascending=False).head(top_n)

    return pool[[
        "Product Name", "Product Brand", "Product Price", "Product Rating",
        "product_type", "category_level_1", "category_level_2", "price_band",
        "text_similarity", "type_overlap", "category2_match", "final_score"
    ]].reset_index(drop=True)

hybrid_recommend_v5("HealthVit Activated Charcoal Powder", top_n=5)

,Product Name,Product Brand,Product Price,Product Rating,product_type,category_level_1,category_level_2,price_band,text_similarity,type_overlap,category2_match,final_score
0,Ancient Living Tea Tree Face Wash,Ancient Living,199,4.0,Acne Treatment,Natural,Shop By Concern,Budget,0.032853,1.0,1.0,0.521747
1,Biotique Bio Morning Nectar Visibly Flawless F...,Biotique,49,4.1,Pigmentation,Natural,Shop By Concern,Budget,0.033038,0.0,1.0,0.333859
2,Lotus Herbals Whiteglow Activated Charcoal Bri...,Lotus Herbals,140,4.2,Face Wash,Natural,Skin,Budget,0.340120,0.0,0.0,0.264642
3,Neemli Naturals Glycolic Acid & Hydrolysed Col...,Neemli Naturals,675,4.5,Pigmentation,Natural,Shop By Concern,Mid-range,0.030083,0.0,1.0,0.227579
4,Lotus Herbals Tea Tree & Cinnamon Anti-acne Oi...,Lotus Herbals,140,4.5,Tea Tree Oil,Natural,Trending Searches,Budget,0.040466,0.0,0.0,0.152038


In [77]:
hybrid_recommend_v5("ETUDE HOUSE Face Color Shading - 02", top_n=5)


,Product Name,Product Brand,Product Price,Product Rating,product_type,category_level_1,category_level_2,price_band,text_similarity,type_overlap,category2_match,final_score
0,L.A. Colors Highlight & Contour Palette - Ligh...,L.A. Colors,325,3.8,Contour,Makeup,Face,Mid-range,0.095452,1.0,1.0,0.562411
1,L.A. Girl Pro Contour Cream - Deep,L.A. Girl,675,4.0,Highlighters,Makeup,Face,Mid-range,0.085879,0.0,1.0,0.361827
2,Makeup Revolution HD Pro Ultra Powder Contour ...,Makeup Revolution,1850,4.2,Highlighters,Makeup,Face,Luxury,0.083160,0.0,1.0,0.254399
3,Smashbox L.A. Lights Blendable Lip & Cheek Col...,Smashbox,2300,4.1,Blush,Makeup,Face,Luxury,0.031465,0.0,1.0,0.233151
4,Nykaa Get Cheeky! Blush Duo Palette - Cali Chi...,Nykaa Cosmetics,629,4.4,Blush | Combos @ Nykaa,Makeup,Face | Combos @ Nykaa,Mid-range,0.037309,0.0,0.0,0.155422


In [78]:
new_weights = {"text": 0.75, "type": 0.10, "category2": 0.05, "price": 0.05, "quality": 0.05}

hybrid_recommend_v5("HealthVit Activated Charcoal Powder", top_n=5, weights=new_weights)

,Product Name,Product Brand,Product Price,Product Rating,product_type,category_level_1,category_level_2,price_band,text_similarity,type_overlap,category2_match,final_score
0,Lotus Herbals Whiteglow Activated Charcoal Bri...,Lotus Herbals,140,4.2,Face Wash,Natural,Skin,Budget,0.340120,0.0,0.0,0.316678
1,Ancient Living Tea Tree Face Wash,Ancient Living,199,4.0,Acne Treatment,Natural,Shop By Concern,Budget,0.032853,1.0,1.0,0.231603
2,Biotique Bio Morning Nectar Visibly Flawless F...,Biotique,49,4.1,Pigmentation,Natural,Shop By Concern,Budget,0.033038,0.0,1.0,0.143771
3,Lotus Herbals Tea Tree & Cinnamon Anti-acne Oi...,Lotus Herbals,140,4.5,Tea Tree Oil,Natural,Trending Searches,Budget,0.040466,0.0,0.0,0.114178
4,Nykaa Naturals Haldi & Chandan Bathing Soap,Nykaa Naturals,100,4.3,Combination Skin,Natural,Types of Skin,Budget,0.049190,0.0,0.0,0.108016


In [79]:
hybrid_recommend_v5("ETUDE HOUSE Face Color Shading - 02", top_n=5, weights=new_weights)

,Product Name,Product Brand,Product Price,Product Rating,product_type,category_level_1,category_level_2,price_band,text_similarity,type_overlap,category2_match,final_score
0,L.A. Colors Highlight & Contour Palette - Ligh...,L.A. Colors,325,3.8,Contour,Makeup,Face,Mid-range,0.095452,1.0,1.0,0.291046
1,L.A. Girl Pro Contour Cream - Deep,L.A. Girl,675,4.0,Highlighters,Makeup,Face,Mid-range,0.085879,0.0,1.0,0.187591
2,Makeup Revolution HD Pro Ultra Powder Contour ...,Makeup Revolution,1850,4.2,Highlighters,Makeup,Face,Luxury,0.083160,0.0,1.0,0.129347
3,Nykaa Get Cheeky! Blush Duo Palette - Cali Chi...,Nykaa Cosmetics,629,4.4,Blush | Combos @ Nykaa,Makeup,Face | Combos @ Nykaa,Mid-range,0.037309,0.0,0.0,0.116614
4,House Of Makeup Double Duty Kohl + Liner - Sil...,House of Makeup,499,2.8,Kajal,Makeup,Eyes,Mid-range,0.081206,0.0,0.0,0.115779


In [80]:
# MESMARA Extra Virgin Olive Oil

hybrid_recommend_v5("MESMARA Extra Virgin Olive Oil", top_n=5, weights=new_weights)

,Product Name,Product Brand,Product Price,Product Rating,product_type,category_level_1,category_level_2,price_band,text_similarity,type_overlap,category2_match,final_score
0,Allin Exporters Jojoba Oil,Allin Exporters,282,4.0,Face Oils,Natural,Skin,Budget,0.077423,1.0,1.0,0.258067
1,MESMARA Citronella Essential Oil,MESMARA,349,3.0,Essential Oils,Natural,Aromatherapy,Mid-range,0.323910,0.0,0.0,0.242933
2,MESMARA Ylang Ylang Essential Oil,MESMARA,339,NaN,Essential Oils,Natural,Aromatherapy,Mid-range,0.301961,0.0,0.0,0.226471
3,Aroma Magic Sunblock Lotion SPF++30 UVA/UVB,Aroma Magic,210,4.0,Sunscreen,Natural,Skin,Budget,0.039852,0.0,1.0,0.164443
4,Parachute Advansed Olive & Almond Body Oil,Parachute,74,4.3,Dry Skin,Natural,Types of Skin,Budget,0.112239,0.0,0.0,0.148029


In [81]:
# okay. locked in.

**PrettyPlease Recommender — v1 to v5 Summary**

**v1 — Plain TF-IDF cosine similarity.** Combined product name, brand, description, tags, and category into one text field, vectorized it with TF-IDF, and recommended the nearest products by cosine similarity alone. Worked okay for the top 2-3 results but degraded fast — items with no real relationship (lipsticks, kajal, serum) showed up in the top 10 just because they shared some vocabulary.

**v2 — Added metadata scoring.** Introduced a weighted blend: 70% text similarity + 15% product type match + 10% price band match + 5% brand match. This was the first hybrid attempt, but two problems showed up:

- `quality_score` (rating × log of review count) was raw and unnormalized, so popular products could dominate rankings regardless of relevance.
- `product_type` matching used exact string equality, which broke on products that had multiple pipe-joined types after deduplication (e.g. `"Blush | Combos @ Nykaa"`).

**v3 — Fixed those two issues.** Normalized `quality_score` to 0–1 with MinMaxScaler so it could only act as a small tiebreaker. Replaced exact type matching with Jaccard overlap on a set of type tokens, so partial matches counted. This surfaced a new problem: almost nothing shared an exact `product_type` with the query, so `type_overlap` was 0 for nearly everyone, and quality score quietly took over the ranking again.

**v4 — Added a broader category signal and category leakage fix.** Added `category_level_2` (a broader bucket like "Face" or "Lips") as its own scoring term, since exact `product_type` was too narrow. But that let skincare products leak into makeup results (both tagged "Face"). Fixed by restricting candidates to the same `category_level_1` first (Makeup vs. Skin vs. Natural, etc.) before scoring anything else.

**v5 — Final tuning and safeguard.** Testing across three different categories (Makeup, Natural/skincare, Natural/oils) showed that `category_level_2` groupings are unreliable outside Makeup — e.g. Natural's "Shop By Concern" bucket grouped a charcoal powder with a totally unrelated tea tree face wash. To stop weak category matches from ever outranking a strong text match, two changes were made:

1. Shifted weight heavily back toward text similarity (0.75) and cut back type/category weights (0.10 / 0.05), since text similarity proved the most consistently trustworthy signal across categories.
2. Added a `min_text_similarity` floor — a candidate needs at least some real textual relationship to the query before category/type overlap can boost it in. This is what fixed the charcoal powder case.

**FINAL WEIGHTS:**

`text_similarity: 0.75`  
`type_overlap: 0.10`  
`category_level_2: 0.05`  
`price_band_match: 0.05`  
`quality_score_norm: 0.05`

### 8.5 Build a query-text function that takes free-text preferences instead of a product name

In [ ]:
def recommend_from_preferences(preference_text, top_n=10, category_filter=None, price_band_filter=None):
    # Turn the user's free-text preference into a TF-IDF vector using the SAME vectorizer
    # that built tfidf_matrix, so it lives in the same feature space
    query_vector = tfidf.transform([preference_text])

    # Compare against every product's text vector
    text_scores = cosine_similarity(query_vector, tfidf_matrix).flatten()

    pool = beauty_df.copy()
    pool_positions = pool.index.to_numpy()
    pool_text_scores = text_scores[pool_positions]

    # Optional hard filters — since this is a fresh user, we can let them
    # narrow the search explicitly instead of inferring it from one product
    if category_filter:
        keep = pool["category_level_1"].astype(str).str.lower() == category_filter.lower()
        pool = pool[keep]
        pool_text_scores = pool_text_scores[keep.to_numpy()]

    if price_band_filter:
        keep = pool["price_band"] == price_band_filter
        pool = pool[keep]
        pool_text_scores = pool_text_scores[keep.to_numpy()]

    if pool.empty:
        filters_applied = []
        if category_filter:
            filters_applied.append(f"category='{category_filter}'")
        if price_band_filter:
            filters_applied.append(f"price_band='{price_band_filter}'")
        filter_msg = " and ".join(filters_applied) if filters_applied else "no filters"
        return f"No products found matching {filter_msg}. Try loosening your filters."

    quality_scores = pool["quality_score_norm"].fillna(0).to_numpy()

    final_scores = 0.85 * pool_text_scores + 0.15 * quality_scores

    pool = pool.assign(text_similarity=pool_text_scores, final_score=final_scores)
    pool = pool.sort_values("final_score", ascending=False).head(top_n)

    return pool[[
        "Product Name", "Product Brand", "Product Price", "Product Rating",
        "product_type", "category_level_1", "price_band",
        "text_similarity", "final_score"
    ]].reset_index(drop=True)

In [83]:
recommend_from_preferences(
    "long-lasting matte lipstick in a deep red shade",
    top_n=5
)

,Product Name,Product Brand,Product Price,Product Rating,product_type,category_level_1,price_band,text_similarity,final_score
0,L'Oreal Paris Color Riche Moist Matte Lipstick...,L'Oreal Paris,375,4.3,Lipstick,Makeup,Mid-range,0.173812,0.272233
1,M.A.C Retro Matte Lipstick - Flat Out Fabulous,M.A.C,1700,4.6,Lipstick,Makeup,Luxury,0.155961,0.267778
2,Nykaa Wonderpuff Cushion Liquid Lipstick - Wer...,Nykaa Cosmetics,539,4.0,Lip Stain,Makeup,Mid-range,0.155838,0.235541
3,Maybelline New York Color Sensational The Load...,Maybelline New York,238,4.4,Lipstick,Makeup,Budget,0.129417,0.231478
4,Masaba By Nykaa Lipstick - Touch Me Not,Nykaa Cosmetics,449,4.2,Lipstick,Makeup,Mid-range,0.108187,0.214135


In [84]:
recommend_from_preferences(
    "hydrating face moisturizer for dry skin",
    top_n=5,
    category_filter="Skin",
    price_band_filter="Budget"
)

,Product Name,Product Brand,Product Price,Product Rating,product_type,category_level_1,price_band,text_similarity,final_score
0,Ponds Super Light Gel Oil Free Moisturiser Wit...,Ponds,149,4.5,Skin Dryness,Skin,Budget,0.120608,0.248289
1,Ponds Light Moisturiser Non-Oily Fresh Feel Wi...,Ponds,120,4.4,Face Moisturizer & Day Cream,Skin,Budget,0.137202,0.244732
2,Bio Oil Dry Skin Gel,Bio Oil,230,4.2,Pore Care,Skin,Budget,0.163818,0.230061
3,NIVEA Soft - Light Moisturising Cream,Nivea,185,4.5,Face Moisturizer & Day Cream,Skin,Budget,0.110030,0.227340
4,Ponds Cold Cream Honey & Milk Protein Face Cream,Ponds,94,4.4,Face Moisturizer & Day Cream,Skin,Budget,0.132625,0.186756


## 9. RAG Implementation
(finally)

In [85]:
pip install google-genai python-dotenv


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [86]:
from dotenv import load_dotenv
import os

load_dotenv()

assert os.getenv("GEMINI_API_KEY"), "GEMINI_API_KEY not found — check your .env file"

In [87]:
from google import genai

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

def rag_recommend(preference_text, top_n=5, category_filter=None, price_band_filter=None):
    retrieved = recommend_from_preferences(
        preference_text, top_n=top_n,
        category_filter=category_filter,
        price_band_filter=price_band_filter
    )

    if isinstance(retrieved, str):
        return retrieved

    context_lines = []
    for _, row in retrieved.iterrows():
        context_lines.append(
            f"- {row['Product Name']} by {row['Product Brand']}, "
            f"₹{row['Product Price']}, rated {row['Product Rating']}★, "
            f"type: {row['product_type']}"
        )
    context = "\n".join(context_lines)

    prompt = f"""A user is looking for: "{preference_text}"

Here are the top matching products retrieved from our catalog:
{context}

Write a short, friendly recommendation (3-5 sentences) explaining which 1-2 products
best fit what they asked for and why. Only reference products listed above — do not
invent products or details not shown."""

    interaction = client.interactions.create(
        model="gemini-3.6-flash",
        input=prompt
    )

    return {
        "retrieved_products": retrieved,
        "recommendation_text": interaction.output_text
    }

In [88]:
result = rag_recommend(
    "hydrating face moisturizer for dry skin",
    top_n=5,
    category_filter="Skin",
    price_band_filter="Budget"
)

print(result["recommendation_text"])
result["retrieved_products"]

If you're looking to hydrate dry skin, we highly recommend the **Ponds Super Light Gel Oil Free Moisturiser With Hyaluronic Acid + Vitamin E** (₹149, rated 4.5★). It is specifically formulated for skin dryness and uses Hyaluronic Acid and Vitamin E to deliver deep hydration. Another excellent pick is the **NIVEA Soft - Light Moisturising Cream** (₹185, rated 4.5★), which serves as a great daily face moisturizer that keeps your skin feeling soft and fresh. Both of these top-rated options offer lightweight, effective moisture to help keep dry skin at bay!


,Product Name,Product Brand,Product Price,Product Rating,product_type,category_level_1,price_band,text_similarity,final_score
0,Ponds Super Light Gel Oil Free Moisturiser Wit...,Ponds,149,4.5,Skin Dryness,Skin,Budget,0.120608,0.248289
1,Ponds Light Moisturiser Non-Oily Fresh Feel Wi...,Ponds,120,4.4,Face Moisturizer & Day Cream,Skin,Budget,0.137202,0.244732
2,Bio Oil Dry Skin Gel,Bio Oil,230,4.2,Pore Care,Skin,Budget,0.163818,0.230061
3,NIVEA Soft - Light Moisturising Cream,Nivea,185,4.5,Face Moisturizer & Day Cream,Skin,Budget,0.110030,0.227340
4,Ponds Cold Cream Honey & Milk Protein Face Cream,Ponds,94,4.4,Face Moisturizer & Day Cream,Skin,Budget,0.132625,0.186756


## 10. Enhancing Explainability + Metrics

In [91]:
# picking up 15-20 query products spanning categories

eval_products = (
    beauty_df
    .dropna(subset=["category_level_1", "product_type"])
    .groupby("category_level_1", group_keys=False)
    .apply(lambda x: x.sample(min(3, len(x)), random_state=42))
    [["Product Name", "category_level_1", "product_type", "price_band"]]
)

eval_products

/var/folders/10/8tq7fk_d2w1_g7907_v29q4c0000gn/T/ipykernel_34873/1956784236.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(3, len(x)), random_state=42))


,Product Name,category_level_1,product_type,price_band
30,Himalaya Herbals Clear Complexion Whitening Fa...,Brand,Himalaya,Budget
36,Biotique Bio Gotu Kola Smooth Skin Lotion,Brand,Biotique,Budget
101,Lotus Herbals Baby + Tender Touch Baby Body Lo...,Brand,Lotus Herbals,Budget
352,Himalaya Stretch Mark Cream For Moms,Brand | Skin,Himalaya | Lotions & Creams,Budget
406,Lotus Herbals Jojobawash Active Milli Capsules...,Brand | Skin,Lotus Herbals | Face Wash,Budget
427,Biotique Bio Papaya Revitalizing Tan removal S...,Brand | Skin,Biotique | Scrubs & Exfoliators,Budget
380,Dunhill Desire Eau De Toilette,Fragrance,Perfumes (EDT & EDP),Luxury
141,Davidoff Cool Water Woman Wave Eau De Toilette,Fragrance,Perfumes (EDT & EDP),Luxury
258,Kenzo World Eau De Parfum,Fragrance,Perfumes (EDT & EDP),Luxury
228,Burberry My Burberry Eau De Toilette,Fragrance | Nykaa Luxe,Perfumes (EDT & EDP) | Perfumes (EDP/EDT),Luxury


In [92]:
# generating top-10 for each and save to a labeling sheet

rows = []

for _, prod in eval_products.iterrows():
    result = hybrid_recommend_v5(prod["Product Name"], top_n=10)
    if isinstance(result, str):
        continue
    for rank, (_, rec) in enumerate(result.iterrows(), start=1):
        rows.append({
            "query_product": prod["Product Name"],
            "query_category": prod["category_level_1"],
            "query_price_band": prod["price_band"],
            "rank": rank,
            "recommended_product": rec["Product Name"],
            "recommended_category": rec["category_level_1"],
            "recommended_price_band": rec["price_band"],
            "relevant": None  # you'll fill this in manually — 1 or 0
        })

eval_df = pd.DataFrame(rows)
eval_df.to_csv("eval_labeling_sheet.csv", index=False)
eval_df.head(15)

/var/folders/10/8tq7fk_d2w1_g7907_v29q4c0000gn/T/ipykernel_34873/880413544.py:5: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  matches = beauty_df[beauty_df["Product Name"].str.contains(product_name, case=False, na=False)]
/var/folders/10/8tq7fk_d2w1_g7907_v29q4c0000gn/T/ipykernel_34873/880413544.py:5: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  matches = beauty_df[beauty_df["Product Name"].str.contains(product_name, case=False, na=False)]
/var/folders/10/8tq7fk_d2w1_g7907_v29q4c0000gn/T/ipykernel_34873/880413544.py:5: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  matches = beauty_df[beauty_df["Product Name"].str.contains(product_name, case=False, na=False)]


,query_product,query_category,query_price_band,rank,recommended_product,recommended_category,recommended_price_band,relevant
0,Himalaya Herbals Clear Complexion Whitening Fa...,Brand,Budget,1,Himalaya Herbals Refreshing Cleansing Milk,Brand,Budget,None
1,Himalaya Herbals Clear Complexion Whitening Fa...,Brand,Budget,2,Himalaya Anti-Hair Fall Shampoo With Bhringaraja,Brand,Budget,None
2,Himalaya Herbals Clear Complexion Whitening Fa...,Brand,Budget,3,Himalaya Baby Hair Oil,Brand,Budget,None
3,Himalaya Herbals Clear Complexion Whitening Fa...,Brand,Budget,4,Himalaya Men Daily Nourish Hair Cream,Brand,Budget,None
4,Himalaya Herbals Clear Complexion Whitening Fa...,Brand,Budget,5,Himalaya Herbals Soothing Body Lotion,Brand,Budget,None
5,Himalaya Herbals Clear Complexion Whitening Fa...,Brand,Budget,6,Nykaa Wanderlust Handwash - Japanese Cherry Bl...,Brand,Budget,None
6,Himalaya Herbals Clear Complexion Whitening Fa...,Brand,Budget,7,Biotique Bio Thyme Volume Conditioner For Fine...,Brand,Budget,None
7,Himalaya Herbals Clear Complexion Whitening Fa...,Brand,Budget,8,VLCC Total Nourishment Fruit Cream,Brand,Budget,None
8,Himalaya Herbals Clear Complexion Whitening Fa...,Brand,Budget,9,Nykaa Home Safe - Disinfectant Floor Cleaner -...,Brand,Budget,None
9,Himalaya Herbals Clear Complexion Whitening Fa...,Brand,Budget,10,Lotus Herbals Baby + Tender Touch Baby Body Lo...,Brand,Budget,None


### 10.1 LLM-Based Relevance Labeling
Originally planned to write an in-notebook LLM judge function (llm_judge_relevance()) to auto-label the (query_product, recommended_product) pairs generated by hybrid_recommend_v5. Due to API quota limits during development, this step was outsourced instead: the unlabeled pairs (eval_labeling_sheet.csv) were exported and labeled externally via ChatGPT using the same relevance criteria the in-notebook judge would have used, producing eval_labeling_sheet_llm_labeled.csv.

In [95]:
import pandas as pd

DATA_DIR = "../data"

eval_df = pd.read_csv(f"{DATA_DIR}/eval_labeling_sheet.csv")
labeled = pd.read_csv(f"{DATA_DIR}/eval_labeling_sheet_llm_labeled.csv")

print("eval_df shape:", eval_df.shape)
print("labeled shape:", labeled.shape)
print(labeled.columns.tolist())

eval_df shape: (211, 8)
labeled shape: (211, 8)
['query_product', 'query_category', 'query_price_band', 'rank', 'recommended_product', 'recommended_category', 'recommended_price_band', 'relevant']


In [96]:
# sanity check
print(labeled["relevant"].value_counts())
print(labeled["relevant"].isnull().sum(), "nulls")

# make sure query_product/recommended_product pairs line up identically between the two sheets
mismatch = (eval_df["query_product"] != labeled["query_product"]).sum() + \
           (eval_df["recommended_product"] != labeled["recommended_product"]).sum()
print("Row mismatches between original and labeled sheet:", mismatch)

relevant
1    172
0     39
Name: count, dtype: int64
0 nulls
Row mismatches between original and labeled sheet: 0


### 10.2 Evaluation

In [97]:
def precision_at_k(df, k):
    scores = []
    for query, group in df.groupby("query_product"):
        top_k = group[group["rank"] <= k]
        precision = top_k["relevant"].sum() / k
        scores.append(precision)
    return sum(scores) / len(scores)

p_at_5 = precision_at_k(labeled, 5)
p_at_10 = precision_at_k(labeled, 10)

print(f"Precision@5:  {p_at_5:.3f}")
print(f"Precision@10: {p_at_10:.3f}")

Precision@5:  0.721
Precision@10: 0.614


In [98]:
def category_purity_at_k(df, k):
    scores = []
    for query, group in df.groupby("query_product"):
        top_k = group[group["rank"] <= k]
        match = (top_k["query_category"] == top_k["recommended_category"]).sum()
        scores.append(match / k)
    return sum(scores) / len(scores)

print(f"Category purity@5:  {category_purity_at_k(labeled, 5):.3f}")
print(f"Category purity@10: {category_purity_at_k(labeled, 10):.3f}")

Category purity@5:  0.886
Category purity@10: 0.754


In [99]:
band_order = ["Budget", "Mid-range", "Premium", "Luxury"]

def band_distance(b1, b2):
    if pd.isna(b1) or pd.isna(b2):
        return None
    return abs(band_order.index(b1) - band_order.index(b2))

def price_consistency_at_k(df, k, max_distance=1):
    scores = []
    for query, group in df.groupby("query_product"):
        top_k = group[group["rank"] <= k].copy()
        top_k["dist"] = top_k.apply(
            lambda r: band_distance(r["query_price_band"], r["recommended_price_band"]), axis=1
        )
        top_k = top_k.dropna(subset=["dist"])
        if len(top_k) == 0:
            continue
        within = (top_k["dist"] <= max_distance).sum()
        scores.append(within / len(top_k))
    return sum(scores) / len(scores)

print(f"Price-band consistency@10 (same or adjacent band): {price_consistency_at_k(labeled, 10):.3f}")

Price-band consistency@10 (same or adjacent band): 0.881


### 10.3 Examples for documentation

In [100]:
# Best examples: queries where ALL top-5 were labeled relevant
perfect_queries = (
    labeled[labeled["rank"] <= 5]
    .groupby("query_product")["relevant"]
    .mean()
)
best_examples = perfect_queries[perfect_queries == 1.0].index.tolist()
print("Perfect top-5 queries:", best_examples[:5])

# Worst examples: queries where precision@5 was lowest
worst_examples = perfect_queries.sort_values().index.tolist()
print("Worst top-5 queries:", worst_examples[:5])

Perfect top-5 queries: ['Biotique Bio Papaya Revitalizing Tan removal Scrub', 'Bobbi Brown Extra Illuminating Moisture Balm - Bare Glow', 'Burberry Brit Rhythm Men Eau De Toilette', 'Cadiveu Brasil Cacau Brazilian Thermal Reconstruction', 'Charlotte Tilbury Matte Revolution - Festival Magic']
Worst top-5 queries: ['Vaadi Herbals Pedicure - Manicure Spa Kit Soothing & Relaxing', 'Himalaya Herbals Clear Complexion Whitening Face Scrub', 'Just Herbs Kumuda Sacred Lotus Rejuvenating Body Wash', 'VLCC Ayurveda Baby Cream', 'Spinz Master Mover Deodorant Body Spray For Men']


In [101]:
example_query = worst_examples[0]  # or any specific product name you want to feature
labeled[
    (labeled["query_product"] == example_query) & (labeled["rank"] <= 5)
][["query_product", "recommended_product", "recommended_category", "relevant"]]

,query_product,recommended_product,recommended_category,relevant
201,Vaadi Herbals Pedicure - Manicure Spa Kit Soot...,VLCC 3 In 1 Intensive Care Cold Cream SPF 20,Skin,0
202,Vaadi Herbals Pedicure - Manicure Spa Kit Soot...,VLCC Ayurveda Baby Cream,Skin,0
203,Vaadi Herbals Pedicure - Manicure Spa Kit Soot...,Estee Lauder Perfectly Clean Multi Action Foam...,Skin,0
204,Vaadi Herbals Pedicure - Manicure Spa Kit Soot...,Bio Oil Dry Skin Gel,Skin,1
205,Vaadi Herbals Pedicure - Manicure Spa Kit Soot...,Olivia Herb Bleach,Skin,0


## 11. Test Cases

### Methodology

Relevance was evaluated using a proxy labeling approach, since no user
interaction data exists for this cold-start recommendation scenario.
18 query products were sampled across the dataset's top-level categories
(Makeup, Skin, Natural, Hair, Fragrance, etc.), and the top-10 recommendations
from `hybrid_recommend_v5` were generated for each (211 query-recommendation
pairs total). Each pair was labeled relevant (1) or not relevant (0) using an
LLM judge, based on whether a customer interested in the query product would
reasonably be happy to see the recommended product suggested.

**Results:**
- Precision@5: 0.721
- Precision@10: 0.614
- Category purity@5: 0.886 / @10: 0.754
- Price-band consistency@10 (same or adjacent band): 0.881

### Successful Scenarios

The system performs strongly when the query product has clear category and
type signals, and when the catalog contains multiple genuinely similar
products to draw from.

**Example: "Bobbi Brown Extra Illuminating Moisture Balm - Bare Glow"**
All 5 top recommendations were judged relevant. The query is a well-defined
Makeup/Face product with specific descriptive language ("illuminating",
"moisture balm"), which gives TF-IDF strong textual signal to work with, and
the catalog contains enough comparable Face products (highlighters, balms,
similar finish products) for the category and type overlap scores to
reinforce good matches rather than compensate for weak ones.

**Example: "Charlotte Tilbury Matte Revolution - Festival Magic"**
All 5 top recommendations were judged relevant. Lipstick is one of the
best-represented product types in the dataset (25+ products), so the
recommender has a dense neighborhood of comparable products to rank
against — this is a case where category density directly translates to
recommendation quality.

**Pattern:** success correlates with (a) well-represented product types in
the dataset, and (b) descriptive, specific product text that TF-IDF can
distinguish from unrelated products.

### Failure Scenarios

**Example: "Vaadi Herbals Pedicure - Manicure Spa Kit"**
Only 1 of 5 top recommendations was judged relevant (Bio Oil Dry Skin Gel).
The other 4 (VLCC Cold Cream, VLCC Baby Cream, Estee Lauder facial cleanser,
Olivia Herb Bleach) all share the `category_level_1 = Skin` label and passed
the price-band filter, but none are functionally similar to a nail/pedicure
kit — they're facial skincare, not nail or foot care.

**Root cause:** the dataset has very few nail/pedicure-adjacent products, so
when text similarity for the true nearest neighbors is weak, the recommender
falls back on the same-category filter, which surfaces *technically same
category, functionally unrelated* products. This is a **coverage gap** in
the underlying dataset, not a scoring logic bug — the `category_level_1`
filter did its job (kept results within Skin), but `category_level_1` is too
coarse to distinguish nail care from facial skincare within that bucket.

**Pattern:** the system struggles most on niche/under-represented product
types where the catalog simply doesn't contain enough genuinely similar
items, causing the category-level fallback to surface same-bucket-but-wrong-
purpose products.

**Note on Recall:** Recall@K was not computed. Recall requires knowing the
total number of relevant items for each query across the entire catalog
(437 products), which would require labeling every query-candidate pair
(437 × 18 ≈ 7,866 pairs) rather than just the retrieved top-10 (211 pairs).
This was out of scope given labeling cost. Precision@K, category purity,
and price-band consistency were used instead as the primary evaluation
metrics, which is a reasonable substitution in a cold-start setting with
no ground-truth relevance data.

### Takeaway

Precision and category purity are both meaningfully higher at @5 than @10
across the evaluation set, confirming that recommendation quality degrades
predictably as text similarity weakens further down the ranked list — the
system is more trustworthy for "show me the single best match" than "show me
10 alternatives," which is worth stating explicitly as a system characteristic
rather than treating uniformly as a defect.